# Лекция 2. Агент, среда, награда, политика. Строим свою среду

**Цели лекции**

1. Научиться раскладывать любую задачу на агента, среду, наблюдение, действие, награду и политику — и понимать, где между ними граница.
2. Понимать, как устроена награда: разреженная и плотная, зачем нужен shaping и как им себе навредить.
3. Уметь написать собственную среду в Gymnasium, проверить её, обернуть и запустить на ней алгоритм из недели 1.

На прошлой лекции мы пользовались готовыми средами. Сегодня откроем капот: посмотрим, что такое среда с точки зрения кода, напишем свою с нуля и научим на ней агента. К концу пары у вас будет шаблон, из которого получается среда для любой задачи с дискретными состояниями — именно это понадобится в домашнем задании и в проектах.

**План и хронометраж (одна пара, 90 минут)**

| | Раздел | Минут |
|---|---|---|
| 1 | Повторение: цикл агент–среда и что мы уже умеем | 5 |
| 2 | Агент и среда: где проходит граница | 7 |
| 3 | Наблюдение и состояние: когда агент видит не всё | 10 |
| 4 | Действия: дискретные, непрерывные, составные | 6 |
| 5 | Награда: разреженная, плотная, shaping и типичные ошибки | 14 |
| 6 | Политика: детерминированная и стохастическая, табличная и параметрическая | 6 |
| 7 | Устройство среды в Gymnasium: `gym.Env`, `spaces`, `reset`, `step` | 10 |
| 8 | Строим свою среду: GridWorld с препятствиями | 12 |
| 9 | Обёртки: изменяем среду, не трогая её код | 8 |
| 10 | Воспроизводимость, векторные среды, регистрация | 7 |
| 11 | Итоги | 5 |

Пометки в заголовках: **📖 дома** — материал для самостоятельного чтения, на лекции можно пропустить; **⏱ если есть время** — резерв.

## 1. Повторение: что мы уже умеем

![loop](../../assets/agent_env_loop.png)

На прошлой неделе мы договорились о словаре: **агент** принимает решения, **среда** отвечает на них наградой и новым состоянием, **политика** $\pi(a \mid s)$ — правило выбора действия, **эпизод** — одна траектория до терминального состояния, **return** $G = \sum_t \gamma^t R_t$ — то, что агент максимизирует. Формально всё это упаковано в **MDP** $\langle \mathcal{S}, \mathcal{A}, P, R, \gamma \rangle$.

И у нас уже есть рабочий алгоритм — метод Cross-Entropy: сыграть много эпизодов, отобрать лучшие, чаще повторять действия из них. Он ничего не знает про устройство среды, ему нужно только `reset()` и `step()`. Сегодня мы посмотрим на среду с другой стороны — со стороны того, кто её **пишет**.

Вопрос залу: вы придумали задачу и хотите решить её с помощью RL. С чего начать — с выбора алгоритма или со среды? Ответ: со среды. Пока не написана среда, алгоритм не на чем запускать; а половина «неудач RL» на практике — это плохо спроектированные наблюдения и награда, а не плохой алгоритм.

## 2. Агент и среда: где проходит граница

Правило простое: **агент — это ровно то, что мы обучаем; среда — всё остальное.** Из него следуют неочевидные вещи.

* **Соперник — часть среды.** В шахматах ходы противника для нашего агента — это стохастические переходы среды. Если соперник тоже учится, среда меняется по ходу обучения (нестационарность) — этим займёмся на неделе 15.
* **Тело робота — часть среды.** Агент — это контроллер, выдающий токи на моторы. Суставы, батарея, задержки датчиков — среда. Поэтому «тот же агент» на другом роботе — другая задача.
* **Интерфейс — часть среды.** Если агент видит игру через 4 кадра, а не через внутреннее состояние движка, у него другая среда, чем у агента с доступом к памяти игры. Разработчик среды решает, что показать.
* **Награда приходит извне.** Агент не может изменить функцию награды — иначе он просто назначит себе бесконечность. Награду задаёт тот, кто ставит задачу (мы), и она живёт в среде.

Практический вывод: **граница агент/среда — это проектное решение**, и сегодня мы будем принимать такие решения сами. Первое из них — что агент видит.

> **Факт.** В DeepMind при обучении агента играть в StarCraft II (AlphaStar) долго спорили, что считать «честным» наблюдением: полная карта из памяти игры или только то, что видит камера человека. Итоговая версия смотрела через камеру и была ограничена в числе действий в минуту — иначе сравнение с людьми теряло смысл. Граница агент/среда определила саму постановку задачи.

## 3. Наблюдение и состояние: когда агент видит не всё

На прошлой лекции мы считали, что агент видит **состояние** $S_t$ — всё, что нужно для решения. На практике агент видит **наблюдение** $O_t$, и оно часто беднее:

* покер: карты соперника скрыты;
* стратегия с туманом войны: видна только часть карты;
* робот с камерой: не видит, что у него за спиной;
* одна картинка из игры: не видно, куда летит мяч.

Задача, где наблюдение не равно состоянию, называется **частично наблюдаемой** (POMDP — Partially Observable MDP). Строгая теория POMDP сложна, но на практике работают два простых приёма:

1. **Добавить историю в наблюдение.** Стек из последних $k$ кадров (так делает DQN в Atari), разность двух последних наблюдений, последние $k$ действий.
2. **Дать агенту память.** Рекуррентная сеть внутри политики (неделя 10).

Убедимся на CartPole, что наблюдение имеет значение. Наша эвристика с прошлой лекции смотрела на угловую скорость шеста $\dot\theta$. Отнимем её у агента.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces

def evaluate(policy, n_episodes=30):
    """policy(obs, prev_obs) -> action. Средний return по эпизодам."""
    env = gym.make("CartPole-v1")
    returns = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=ep)
        prev, total = obs.copy(), 0.0
        while True:
            a = policy(obs, prev)
            prev = obs.copy()
            obs, r, terminated, truncated, _ = env.step(a)
            total += r
            if terminated or truncated:
                break
        returns.append(total)
    env.close()
    return np.mean(returns)

full_obs   = lambda obs, prev: int(obs[3] > 0)               # видим угловую скорость
angle_only = lambda obs, prev: int(obs[2] > 0)               # видим только угол
two_frames = lambda obs, prev: int(obs[2] - prev[2] > 0)     # восстанавливаем скорость из двух кадров

for name, pi in [("полное наблюдение (угол + скорость)", full_obs),
                 ("только угол", angle_only),
                 ("только угол, но два кадра подряд", two_frames)]:
    print(f"{name:38s}: средний return {evaluate(pi):6.1f}")

Одна и та же среда, одна и та же идея политики: без скорости агент проигрывает впятеро, а стоит дать ему два кадра вместо одного — скорость восстанавливается из разности, и результат возвращается. Марковское свойство нарушено не средой, а **нашим выбором наблюдения**, и починили мы его тоже сами.

Правило проектирования наблюдения: **в наблюдении должно быть всё, от чего зависит правильное решение, и по возможности ничего лишнего.** Лишнее не смертельно, но замедляет обучение: агенту придётся самому понять, что цвет неба на решение не влияет.

## 4. Действия: дискретные, непрерывные, составные

Действие — то, чем агент влияет на среду. Форма действий определяет, какие алгоритмы применимы:

| Тип | Примеры | Пространство в Gymnasium | Алгоритмы |
|---|---|---|---|
| **Дискретные** | 4 направления, 18 кнопок Atari, ход в шахматах | `Discrete(n)` | всё табличное, DQN, PPO |
| **Непрерывные** | момент на суставе, угол руля, объём заявки | `Box(low, high, shape)` | policy gradient, DDPG/TD3, PPO |
| **Составные** | руль + газ, «куда» + «какую единицу» | `MultiDiscrete`, `Dict`, `Tuple` | обычно сводят к одному из двух |

Две практические тонкости:

* **Допустимые действия зависят от состояния.** В шахматах легальны не все ходы; в игре с инвентарём нельзя продать то, чего нет. Стандартный приём — **маска действий**: среда возвращает в `info` булев вектор, а агент обнуляет вероятности недопустимых действий.
* **Дискретизация непрерывного** — законный ход, мы уже делали его с Mountain Car. Обратный ход тоже бывает: непрерывный «параметр» дискретного действия.

Правило: **действие — это то, чем вы реально управляете, на том уровне, на котором принимаете решение.** Агент для торговли не решает «купить акцию за 100.37» — он решает «купить 1% портфеля», а исполнение заявки берёт на себя среда.

In [ ]:
# Пространства Gymnasium описывают форму данных и умеют генерировать примеры.
examples = {
    "Discrete(4)":                    spaces.Discrete(4),
    "Box(-1, 1, shape=(2,))":         spaces.Box(-1.0, 1.0, shape=(2,), dtype=np.float32),
    "MultiDiscrete([3, 5])":          spaces.MultiDiscrete([3, 5]),
    "Dict(move=Discrete(4), fire=Discrete(2))":
        spaces.Dict({"move": spaces.Discrete(4), "fire": spaces.Discrete(2)}),
}
for name, space in examples.items():
    space.seed(0)
    print(f"{name:42s} -> пример: {space.sample()}")

# Пространство умеет проверять, что значение ему принадлежит:
box = examples["Box(-1, 1, shape=(2,))"]
print("\n[0.5, -0.2] в Box?", box.contains(np.array([0.5, -0.2], dtype=np.float32)),
      "  [2, 0] в Box?", box.contains(np.array([2.0, 0.0], dtype=np.float32)))

## 5. Награда: разреженная, плотная, shaping и типичные ошибки

Награда — единственный канал, по которому мы объясняем агенту, чего хотим. На прошлой лекции было правило «награда описывает *что*, а не *как*». Сегодня посмотрим, как это правило уживается с практикой.

### 5.1 Разреженная и плотная

![reward](../../assets/reward_types.png)

* **Разреженная** (sparse): почти всегда 0, что-то ненулевое только в конце — «дошёл / не дошёл», «выиграл / проиграл». Честно описывает цель, но пока агент ни разу не дошёл, все эпизоды одинаковы, и учиться не на чем. Мы видели это на Mountain Car.
* **Плотная** (dense): подсказка на каждом шаге — расстояние до цели, скорость, штраф за время. Учиться легко, но теперь агент оптимизирует **подсказку**, а не цель, и может найти способ собирать подсказки, не решая задачу.

### 5.2 Reward shaping

**Shaping** — добавление к настоящей награде вспомогательного слагаемого $F$, которое делает сигнал плотнее:

$$
R'(s, a, s') = R(s, a, s') + F(s, a, s').
$$

Опасность: неудачный $F$ меняет то, что агент считает оптимальным. Классический пример — награда за приближение к цели: агент открывает, что можно бесконечно ходить туда-сюда вокруг цели и собирать «приближения», не заходя в неё.

Есть форма $F$, которая **гарантированно не меняет оптимальную политику** — потенциальный shaping (Ng, Harada, Russell, 1999):

$$
F(s, s') = \gamma\,\Phi(s') - \Phi(s),
$$

где $\Phi(s)$ — любая функция состояния («потенциал», например минус расстояние до цели). Почему работает: на любой траектории эти слагаемые схлопываются в телескопическую сумму, и return меняется на константу $-\Phi(s_0)$, одинаковую для всех траекторий из одного старта. Ходить кругами бесполезно: вернулся в ту же клетку — сумма подсказок обнулилась.

### 5.3 Типичные ошибки

1. **Положительная награда за каждый шаг** («+1 за то, что жив») в задаче, которую нужно закончить: агент учится **не заканчивать** — тянуть время выгоднее, чем дойти до цели. Для CartPole это правильно (цель — продержаться), для лабиринта — катастрофа.
2. **Награда за средство, а не за цель**: «держи угол шеста маленьким» вместо «не урони». Агент найдёт способ держать угол маленьким, необязательно тот, который вы ожидали.
3. **Несбалансированные слагаемые**: штраф за энергию в 100 раз больше награды за прогресс — агент выбирает вообще не двигаться. Всегда смотрите на *масштаб* каждого слагаемого на реальной траектории.
4. **Награда, которую можно получить, ничего не делая**: бонус «за исследование новой клетки», не убывающий со временем, — агент бегает по кругу.
5. **Разные награды при обучении и при оценке** без проверки: обучили на плотной, отчитались по плотной, а по настоящей — ноль.

> **Факт.** Программа Тома Мёрфи (2013), игравшая в NES-игры по счёту, в Tetris научилась ставить игру на паузу перед неизбежным проигрышем: на паузе отрицательная награда никогда не наступает. Десятки похожих историй собраны в подборке DeepMind [«Specification gaming: the flip side of AI ingenuity»](https://deepmind.google/discover/blog/specification-gaming-the-flip-side-of-ai-ingenuity/) — почти все они начинаются с плохо спроектированной награды.

Правило: **обучать можно на плотной награде, но оценивать — только по настоящей.** Ниже, в разделе 8, мы построим среду с разреженной наградой, а в разделе 9 добавим к ней потенциальный shaping обёрткой и сравним обучение — с честной оценкой по настоящей цели.

## 6. Политика: четыре варианта

Политика — то, что мы обучаем. Два независимых выбора:

**Детерминированная или стохастическая.** $a = \pi(s)$ выдаёт одно действие; $\pi(a \mid s)$ — распределение. Стохастическая нужна во время обучения (исследование) и в играх с соперником (детерминированного игрока можно «прочитать»). После обучения политику часто делают детерминированной — берут $\arg\max_a \pi(a \mid s)$, хотя мы видели на Frozen Lake, что это не всегда безопасно.

**Табличная или параметрическая.** Таблица $\pi[s, a]$ работает, пока состояний немного, и это всё, что мы использовали до сих пор. Когда состояние — картинка или вектор вещественных чисел, таблицу заменяют функцией с параметрами $\theta$:

$$
\pi_\theta(a \mid s) = \text{softmax}\big(f_\theta(s)\big)_a,
$$

где $f_\theta$ — линейная модель или нейросеть. Обучение сводится к настройке $\theta$ градиентом — как в обычном обучении с учителем. Это неделя 5.

| | Табличная | Параметрическая |
|---|---|---|
| Состояния | конечные, немного | любые |
| Обобщение на невиденные состояния | нет | да |
| Что обучаем | числа в клетках | веса $\theta$ |
| Сегодня | GridWorld, Cross-Entropy | — |

**Эпизодические и непрерывные задачи.** В эпизодической есть конец и `reset()`, в непрерывной — нет, и без $\gamma < 1$ сумма наград бесконечна. Многие «непрерывные» задачи на практике режут на эпизоды искусственно (`TimeLimit`), — и вот тут важна разница `terminated` / `truncated` из прошлой лекции: обрыв по времени не означает, что будущих наград нет.

## 7. Устройство среды в Gymnasium

[Gymnasium](https://gymnasium.farama.org/) — стандартный интерфейс среды, который понимают все библиотеки RL. Среда — класс, наследующий `gym.Env`, с двумя атрибутами и двумя методами:

![anatomy](../../assets/env_anatomy.png)

* `observation_space`, `action_space` — **описание формы** наблюдений и действий. Алгоритм по ним понимает, какой размер входа и выхода делать у сети.
* `reset(seed=None)` → `(obs, info)` — начать эпизод. Если передан `seed`, среда инициализирует `self.np_random` — свой генератор случайных чисел, которым должна пользоваться вся случайность внутри среды.
* `step(action)` → `(obs, reward, terminated, truncated, info)` — один шаг. `terminated` — среда пришла в терминальное состояние, `truncated` — эпизод оборван снаружи. `info` — словарь для всего остального (маска действий, диагностика).
* `render()` — картинка или текст для человека; режим задаётся при создании (`render_mode="rgb_array"` возвращает массив пикселей).

Посмотрим на готовую среду изнутри. `gym.make` возвращает не сам класс среды, а **луковицу из обёрток** вокруг него:

In [ ]:
env = gym.make("CartPole-v1")
layer = env
while hasattr(layer, "env"):                 # разворачиваем луковицу
    print(type(layer).__name__, end=" -> ")
    layer = layer.env
print(type(layer).__name__, "(это и есть env.unwrapped)")

print("\nspec.max_episode_steps =", env.spec.max_episode_steps)
print("observation_space      =", env.observation_space)
print("action_space           =", env.action_space)

obs, info = env.reset(seed=0)
obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
print("\nstep ->", obs.round(3), reward, terminated, truncated, info)
env.close()

Три слоя `gym.make` добавляет сам: `PassiveEnvChecker` проверяет типы, `OrderEnforcing` следит, чтобы `step()` не вызывали до `reset()`, `TimeLimit` обрывает эпизод на 500-м шаге с `truncated=True`. Сама физика тележки живёт в `CartPoleEnv` и доступна как `env.unwrapped`. К обёрткам вернёмся в разделе 9 — а сейчас напишем свою среду.

## 8. Строим свою среду: GridWorld с препятствиями

Задача: сетка, стены, старт и цель. Агент ходит в четыре стороны, с вероятностью `slip` его «заносит» в случайную сторону. Награда — 1 за приход в цель, иначе 0 (разреженная, как в Frozen Lake). Состояние — номер клетки.

Проектные решения, которые мы принимаем прямо сейчас (в домашнем задании их будете принимать вы):

| Вопрос | Наше решение | Альтернатива |
|---|---|---|
| Что такое состояние? | номер клетки, `Discrete(rows*cols)` | координаты `Box`, картинка |
| Что такое действие? | `Discrete(4)` | `MultiDiscrete` (направление + сколько шагов) |
| Где случайность? | `slip`: действие заменяется случайным | случайное появление стен, ветер |
| Когда конец? | пришли в цель → `terminated` | лимит шагов → `truncated` (добавим обёрткой) |
| Какая награда? | разреженная: 1 в цели | плотная (добавим обёрткой) |

Карта задаётся строками: `S` — старт, `G` — цель, `#` — стена, `.` — пол.

In [ ]:
class GridWorldEnv(gym.Env):
    """Сетка с препятствиями. Агент идёт от S к G; с вероятностью slip действие заменяется случайным."""

    metadata = {"render_modes": ["ansi", "rgb_array"], "render_fps": 4}
    MOVES = {0: (0, -1), 1: (1, 0), 2: (0, 1), 3: (-1, 0)}   # ←, ↓, →, ↑  как (dr, dc)
    ARROWS = "←↓→↑"

    DEFAULT_LAYOUT = [
        "S....#",
        ".##..#",
        "...#..",
        ".#..#.",
        ".#.#..",
        "....#G",
    ]

    def __init__(self, layout=None, slip=0.0, render_mode=None):
        self.layout = [list(row) for row in (layout or self.DEFAULT_LAYOUT)]
        self.n_rows, self.n_cols = len(self.layout), len(self.layout[0])
        self.slip = slip
        self.render_mode = render_mode

        self.observation_space = spaces.Discrete(self.n_rows * self.n_cols)
        self.action_space = spaces.Discrete(4)

        self.start = self._find("S")
        self.goal = self._find("G")
        self.pos = self.start

    def _find(self, char):
        for r, row in enumerate(self.layout):
            for c, cell in enumerate(row):
                if cell == char:
                    return (r, c)
        raise ValueError(f"на карте нет клетки {char!r}")

    def _obs(self):
        return self.pos[0] * self.n_cols + self.pos[1]        # номер клетки

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)              # инициализирует self.np_random
        self.pos = self.start
        return self._obs(), {}

    def step(self, action):
        if self.slip > 0 and self.np_random.random() < self.slip:
            action = int(self.np_random.integers(4))         # поскользнулись
        dr, dc = self.MOVES[int(action)]
        r, c = self.pos[0] + dr, self.pos[1] + dc
        if 0 <= r < self.n_rows and 0 <= c < self.n_cols and self.layout[r][c] != "#":
            self.pos = (r, c)                                 # иначе упёрлись в стену
        terminated = self.pos == self.goal
        reward = 1.0 if terminated else 0.0
        return self._obs(), reward, terminated, False, {}

    def render(self):
        if self.render_mode == "ansi":
            rows = []
            for r, row in enumerate(self.layout):
                rows.append("".join("A" if (r, c) == self.pos else ch for c, ch in enumerate(row)))
            return "\n".join(rows)
        if self.render_mode == "rgb_array":
            return self._render_rgb()

    def _render_rgb(self):
        colors = {".": "#f4f4f4", "#": "#555555", "S": "#cfe3f7", "G": "#a9dfa9"}
        fig, ax = plt.subplots(figsize=(2.6, 2.6 * self.n_rows / self.n_cols))
        for r, row in enumerate(self.layout):
            for c, ch in enumerate(row):
                ax.add_patch(plt.Rectangle((c, r), 1, 1, color=colors[ch], ec="white"))
        ax.add_patch(plt.Circle((self.pos[1] + 0.5, self.pos[0] + 0.5), 0.3, color="#4C72B0"))
        ax.set_xlim(0, self.n_cols); ax.set_ylim(self.n_rows, 0); ax.set_aspect("equal"); ax.axis("off")
        fig.tight_layout(pad=0)
        fig.canvas.draw()
        img = np.asarray(fig.canvas.buffer_rgba())[:, :, :3].copy()
        plt.close(fig)
        return img

In [ ]:
env = GridWorldEnv(render_mode="ansi")
obs, info = env.reset(seed=0)
print("наблюдение:", obs, " пространство:", env.observation_space, env.action_space)
print(env.render())

# Первое, что делают с новой средой — прогоняют проверку интерфейса.
from gymnasium.utils.env_checker import check_env
check_env(GridWorldEnv())
print("\ncheck_env: ошибок нет")

`check_env` проверяет, что `reset` и `step` возвращают то, что обещают пространства, что `seed` действительно делает среду воспроизводимой, что `render` не падает. Это дешёвая страховка от самых обидных багов — например, когда `step` возвращает `numpy.int64` там, где обещан `int`, и алгоритм падает через час обучения.

Теперь картинка вместо текста и случайный агент.

In [ ]:
from matplotlib import animation
from IPython.display import HTML, display

def show_frames(frames, title="", interval=250, width=3.0):
    """Кадры среды -> анимация в ноутбуке (при запуске; в превью на GitHub не видна)."""
    h, w = frames[0].shape[:2]
    fig, ax = plt.subplots(figsize=(width, width * h / w))
    ax.axis("off"); ax.set_title(title, fontsize=10)
    im = ax.imshow(frames[0])
    plt.close(fig)
    anim = animation.FuncAnimation(fig, lambda i: (im.set_data(frames[i]),),
                                   frames=len(frames), interval=interval, blit=True)
    return HTML(anim.to_jshtml(default_mode="loop"))


env = GridWorldEnv(render_mode="rgb_array")
obs, _ = env.reset(seed=0)
frames, total = [env.render()], 0.0
for t in range(40):
    obs, r, terminated, truncated, _ = env.step(env.action_space.sample())
    frames.append(env.render()); total += r
    if terminated or truncated:
        break
print(f"случайный агент: {len(frames) - 1} шагов, return {total}")
show_frames(frames, "Случайный агент в GridWorld")

### Учим агента на своей среде

Алгоритм из недели 1 не знает, что среда наша: ему нужны только `reset`, `step` и размеры пространств. Скопируем его без изменений.

In [ ]:
def run_session(env, policy, rng, max_steps=100):
    """Один эпизод политикой-таблицей: состояния, действия, суммарная награда."""
    obs, _ = env.reset(seed=int(rng.integers(1_000_000)))
    states, actions, total = [], [], 0.0
    for _ in range(max_steps):
        a = int(rng.choice(policy.shape[1], p=policy[obs]))
        states.append(obs); actions.append(a)
        obs, r, terminated, truncated, _ = env.step(a)
        total += r
        if terminated or truncated:
            break
    return states, actions, total


def cross_entropy_method(env, n_iter=25, n_sessions=200, q=0.7, laplace=0.5, mix=0.5,
                         seed=0, max_steps=100, evaluate=None):
    """Табличный Cross-Entropy из лекции 1. evaluate(policy) -> число, если хотим отдельную метрику."""
    rng = np.random.default_rng(seed)
    n_states, n_actions = env.observation_space.n, env.action_space.n
    policy = np.ones((n_states, n_actions)) / n_actions
    log = {"mean": [], "eval": []}
    for it in range(n_iter):
        sessions = [run_session(env, policy, rng, max_steps) for _ in range(n_sessions)]
        returns = np.array([G for _, _, G in sessions])
        threshold = np.quantile(returns, q)
        elite = [s for s in sessions if s[2] >= threshold and s[2] > returns.min()]
        counts = np.full((n_states, n_actions), laplace)
        for states, actions, _ in elite:
            for s, a in zip(states, actions):
                counts[s, a] += 1
        new_policy = policy.copy()
        seen = counts.sum(axis=1) > 0
        new_policy[seen] = counts[seen] / counts[seen].sum(axis=1, keepdims=True)
        policy = mix * new_policy + (1 - mix) * policy
        log["mean"].append(returns.mean())
        if evaluate is not None:
            log["eval"].append(evaluate(policy))
    return policy, log


env = GridWorldEnv()
policy, log = cross_entropy_method(env, n_iter=20, n_sessions=200)

plt.plot(log["mean"], marker=".")
plt.xlabel("итерация"); plt.ylabel("доля успешных эпизодов")
plt.title("Cross-Entropy на нашей среде"); plt.show()

print("выученная политика (самое вероятное действие):")
for r in range(env.n_rows):
    print("  " + " ".join(
        env.layout[r][c] if env.layout[r][c] in "#G" else env.ARROWS[int(np.argmax(policy[r * env.n_cols + c]))]
        for c in range(env.n_cols)))

In [ ]:
# Эпизод выученной политикой, действия — случайно из pi(.|s).
rng = np.random.default_rng(1)
env = GridWorldEnv(render_mode="rgb_array")
obs, _ = env.reset(seed=0)
frames = [env.render()]
for _ in range(50):
    obs, r, terminated, truncated, _ = env.step(int(rng.choice(4, p=policy[obs])))
    frames.append(env.render())
    if terminated or truncated:
        break
show_frames(frames, f"Выученная политика: цель за {len(frames) - 1} шагов")

Тридцать строк — и у нас среда, на которой работает настоящий RL-алгоритм. Всё, что мы дальше будем делать с этой средой, — менять её поведение, **не трогая класс**.

## 9. Обёртки: изменяем среду, не трогая её код

![onion](../../assets/wrapper_onion.png)

**Обёртка** (wrapper) — объект, который выглядит как среда, но внутри держит другую среду и что-то меняет по дороге: подменяет наблюдение, награду, действие или следит за эпизодами. Обёртки можно вкладывать друг в друга; `env.unwrapped` всегда даёт самую внутреннюю среду.

Готовые обёртки, которые понадобятся чаще всего:

| Обёртка | Что делает |
|---|---|
| `TimeLimit(env, max_episode_steps)` | обрывает эпизод, `truncated=True` |
| `RecordEpisodeStatistics(env)` | в конце эпизода кладёт в `info["episode"]` return и длину |
| `TransformReward(env, f)` / `TransformObservation` | применяет функцию к награде / наблюдению |
| `NormalizeObservation`, `NormalizeReward` | бегущая нормализация — важно для нейросетей |
| `FrameStackObservation(env, k)` | стек последних $k$ наблюдений — лекарство от POMDP из раздела 3 |
| `RescaleAction`, `ClipAction` | приводит непрерывные действия к диапазону среды |

Свою обёртку пишут, наследуя `gym.Wrapper` (общий случай) или `gym.RewardWrapper` / `gym.ObservationWrapper` / `gym.ActionWrapper` (переопределить один метод). Сделаем две: лимит шагов из коробки и **потенциальный shaping** своими руками.

In [ ]:
from gymnasium.wrappers import TimeLimit, RecordEpisodeStatistics

class PotentialShaping(gym.Wrapper):
    """Плотная награда: R' = R + gamma * Phi(s') - Phi(s), Phi = -расстояние до цели / scale."""

    def __init__(self, env, gamma=0.99, scale=10.0, step_penalty=0.01):
        super().__init__(env)
        self.gamma, self.scale, self.step_penalty = gamma, scale, step_penalty

    def _phi(self, obs):
        r, c = divmod(int(obs), self.unwrapped.n_cols)
        gr, gc = self.unwrapped.goal
        return -(abs(r - gr) + abs(c - gc)) / self.scale

    def step(self, action):
        phi_before = self._phi(self.unwrapped._obs())
        obs, reward, terminated, truncated, info = self.env.step(action)
        info["true_reward"] = reward                       # настоящую награду сохраняем
        shaped = reward - self.step_penalty + self.gamma * self._phi(obs) - phi_before
        return obs, shaped, terminated, truncated, info


env = RecordEpisodeStatistics(TimeLimit(PotentialShaping(GridWorldEnv()), max_episode_steps=30))
obs, _ = env.reset(seed=0)
while True:
    obs, r, terminated, truncated, info = env.step(env.action_space.sample())
    if terminated or truncated:
        break
print("эпизод закончился:", "terminated" if terminated else "truncated (лимит 30 шагов)")
print("статистика из info['episode']:", {k: float(v) for k, v in info["episode"].items()})
print("самая внутренняя среда:", type(env.unwrapped).__name__)

### Эксперимент: разреженная против плотной

Возьмём карту побольше и жёсткий лимит в 30 шагов: случайный агент почти никогда не успевает дойти. Обучим Cross-Entropy на разреженной награде и на плотной (shaping), а **оценивать** обе политики будем честно — долей эпизодов, в которых агент дошёл до цели без подсказок.

In [ ]:
BIG_LAYOUT = [
    "S.......",
    ".####.#.",
    ".#....#.",
    ".#.##.#.",
    ".#.#..#.",
    ".#.#.##.",
    "...#....",
    "##.#.##G",
]

T = 30
eval_env = TimeLimit(GridWorldEnv(layout=BIG_LAYOUT), max_episode_steps=T)
eval_rng = np.random.default_rng(123)

def true_success(policy, n=200):
    """Доля эпизодов, где агент дошёл до цели, — настоящая цель задачи."""
    return np.mean([run_session(eval_env, policy, eval_rng, max_steps=T + 1)[2] for _ in range(n)])

uniform = np.ones((eval_env.observation_space.n, 4)) / 4
print(f"случайная политика доходит до цели за {T} шагов в {true_success(uniform, 500):.1%} эпизодов")

plt.figure(figsize=(7, 4))
for label, make_env, color in [
    ("разреженная награда", lambda: TimeLimit(GridWorldEnv(layout=BIG_LAYOUT), max_episode_steps=T), "C3"),
    ("плотная: потенциальный shaping", lambda: TimeLimit(PotentialShaping(GridWorldEnv(layout=BIG_LAYOUT)), max_episode_steps=T), "C0"),
]:
    for seed in range(2):
        _, log = cross_entropy_method(make_env(), n_iter=25, n_sessions=200, seed=seed,
                                      max_steps=T + 1, evaluate=true_success)
        plt.plot(log["eval"], color=color, alpha=0.8, label=label if seed == 0 else None)
plt.xlabel("итерация"); plt.ylabel("доля успехов по настоящей награде")
plt.title(f"GridWorld 8×8, лимит {T} шагов"); plt.legend(); plt.show()

С разреженной наградой обучение не начинается вовсе: за 30 шагов случайный агент до цели не доходит, элита пуста. С плотной — агент доходит в трёх четвертях эпизодов, и это измерено по **настоящей** цели, а не по подсказкам. Потенциальный shaping здесь ровно то, чем должен быть: подсказка, которая помогает найти дорогу, но не подменяет цель.

**📖 Дома**: поднимите лимит до 50 шагов и повторите. Разреженная награда начнёт работать (успехи у случайного агента появляются), а плотная выйдет на плато ниже неё. Shaping не меняет *оптимальную* политику, но меняет то, какие эпизоды Cross-Entropy считает элитными, — и это уже свойство алгоритма, а не награды. Подумайте, почему, и проверьте гипотезу, отключив `step_penalty`.

## 10. Воспроизводимость, векторные среды, регистрация

### Сиды

В RL три источника случайности, и у каждого свой генератор: **среда** (`env.reset(seed=...)` → `env.np_random`), **агент** (`np.random.default_rng(seed)` для выбора действий) и **пространства** (`env.action_space.seed(...)` для `sample()`). Эксперимент воспроизводим, только если зафиксированы все три — и он всё равно **не** воспроизводим между версиями библиотек и железом, если в деле нейросети на GPU. Практическое правило: любой график в отчёте — среднее по нескольким сидам, а не один запуск.

### Векторные среды

Один вызов `step()` — один шаг одной среды. Чтобы собирать опыт быстрее, запускают $n$ копий среды и делают шаг во всех сразу: `gym.make_vec`. Наблюдения приходят массивом формы `(n, ...)`, действия тоже подаются массивом. Так устроен сбор данных во всех deep-RL-библиотеках, и с недели 5 мы будем пользоваться этим постоянно.

In [ ]:
import time

# Воспроизводимость: два запуска с одним сидом дают одинаковые траектории.
def rollout(seed):
    env = GridWorldEnv(slip=0.3)
    rng = np.random.default_rng(seed)
    obs, _ = env.reset(seed=seed)
    path = [obs]
    for _ in range(15):
        obs, *_ = env.step(int(rng.integers(4)))
        path.append(obs)
    return path
print("seed=7 дважды одинаково:", rollout(7) == rollout(7), "  seed=7 vs seed=8:", rollout(7) == rollout(8))

# Векторная среда: 8 копий CartPole делают шаг одновременно.
venv = gym.make_vec("CartPole-v1", num_envs=8, vectorization_mode="sync")
obs, _ = venv.reset(seed=0)
print("\nформа наблюдения:", obs.shape, " действий:", venv.action_space)
t0 = time.time()
for _ in range(500):
    obs, rewards, terminated, truncated, info = venv.step(venv.action_space.sample())
print(f"8 сред × 500 шагов за {time.time() - t0:.2f} с; награды за последний шаг: {rewards}")
venv.close()

Тонкость векторных сред: эпизоды в копиях заканчиваются в разное время, и среда **сама** делает `reset` закончившейся копии на следующем шаге. Поэтому флаги `terminated` / `truncated` нужно читать внимательно — это источник половины багов в самописных PPO.

### Регистрация

Чтобы своя среда создавалась через `gym.make("...")` — как встроенные, с `TimeLimit` и проверками, — её регистрируют по имени:

In [ ]:
gym.register(id="GridWorld-v0", entry_point=GridWorldEnv, max_episode_steps=50)

env = gym.make("GridWorld-v0", slip=0.1, render_mode="ansi")
obs, _ = env.reset(seed=0)
print(type(env).__name__, "->", type(env.unwrapped).__name__, "| лимит шагов:", env.spec.max_episode_steps)
print(env.render())

## 11. Итоги

1. **Граница агент/среда — проектное решение.** Агент — то, что мы обучаем; соперник, тело робота, интерфейс и награда — среда.
2. **Наблюдение должно содержать всё, от чего зависит решение.** Если среда частично наблюдаема, добавьте историю (стек кадров) или память.
3. **Награда — это цель, а не подсказка.** Плотная награда и shaping ускоряют обучение, потенциальный shaping $\gamma\Phi(s') - \Phi(s)$ делает это безопасно, но оценивать политику можно только по настоящей награде.
4. **Среда в Gymnasium — это два пространства и два метода.** `check_env` — обязательный первый тест; обёртки меняют поведение среды, не трогая её код; `gym.register` делает свою среду неотличимой от встроенных.

Словарь: **наблюдение и состояние, POMDP, маска действий, разреженная и плотная награда, shaping, потенциальная функция, табличная и параметрическая политика, `gym.Env`, `spaces`, `terminated`/`truncated`, обёртка, векторная среда, сид**.

## Литература

* [Gymnasium: Basic Usage](https://gymnasium.farama.org/introduction/basic_usage/) и [Make your own custom environment](https://gymnasium.farama.org/introduction/create_custom_env/) — официальный туториал, повторяющий раздел 8.
* A. Ng, D. Harada, S. Russell. [Policy invariance under reward transformations](https://people.eecs.berkeley.edu/~pabbeel/cs287-fa09/readings/NgHaradaRussell-shaping-ICML1999.pdf) (ICML 1999) — потенциальный shaping.
* R. Sutton, A. Barto. *Reinforcement Learning: An Introduction*, глава 3 (агент–среда, награда, эпизодические и непрерывные задачи) и раздел 17.4 про проектирование награды.
* DeepMind. [Specification gaming: the flip side of AI ingenuity](https://deepmind.google/discover/blog/specification-gaming-the-flip-side-of-ai-ingenuity/) — коллекция ошибок в награде.

## На семинаре и дома

* Семинар (`../seminar/seminar.ipynb`): пишем GridWorld с нуля по шагам, `check_env`, ручной агент, свои обёртки, Cross-Entropy на своей среде и на скользкой версии.
* ДЗ (`../homework/homework.ipynb`): своя среда «управление запасами» как `gym.Env`, две версии награды и сравнение обучения, формальное описание среды как MDP.